# WM-PeleeNet · 本地模型对照

在相同的 38 类划分上，比较 WM-PeleeNet 重建模型与 YOLO26n。

**前置条件**：先完成本地 YOLO26n 的三组训练、冻结与测试；保留默认 `yolo26n_v1_seed42_manual` 结果目录及权重。第 1 节会核验这些文件，缺失时不能继续。

**流程**：环境与路径 → 数据核验 → 网络检查 → 训练 → 冻结 → 测试 → 模型与速度对照。所有运行开关默认关闭。

模型根据[论文](https://doi.org/10.1049/cit2.12126)重建，并非作者源码；不含论文的中值滤波或自编码器增强。使用 E1 几何增强、224 输入、seed=42 和 AdamW；本模型随机初始化，YOLO 使用预训练权重，训练条件并不完全相同。

## 0. 环境与实验配置

选择 `wafer-yolo26` 内核。若未显示，在激活该环境的终端执行：

```sh
python -m ipykernel install --user --name wafer-yolo26 --display-name "Python (wafer-yolo26)"
```

从仓库或子目录启动 Jupyter；下方自动定位根目录。新实验设置新的 `RUN_TAG`，恢复时保持原配置。

In [ ]:
from pathlib import Path
import os, re, sys

PROJECT = Path.cwd().resolve()
while not (PROJECT / "yolo26s_cls/scripts/prepare_data.py").is_file() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
assert (PROJECT / "yolo26s_cls/scripts/prepare_data.py").is_file(), "请从仓库目录或其子目录启动 Jupyter"
RUN_TAG = "wm_pelee_rebuild_e1_seed42"
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_TAG):
    raise ValueError("RUN_TAG 只能使用字母、数字、下划线和连字符。")
RUN = PROJECT / "wm_peleenet_cls" / "runs" / RUN_TAG
os.environ.setdefault("YOLO_CONFIG_DIR", str(PROJECT / "yolo26s_cls/environment/ultralytics"))
sys.path.insert(0, str(PROJECT / "yolo26s_cls/scripts"))
sys.path.insert(0, str(PROJECT / "wm_peleenet_cls/src"))

import torch
import pandas as pd
import wm_experiment as experiment
from wm_pelee import WMPeleeNet

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu")
CONFIG = {"seed": 42, "epochs": 100, "patience": 15, "batch": 32,
          "learning_rate": 0.001, "weight_decay": 0.0005, "device": str(DEVICE),
          "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU"}
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "设备:", CONFIG["gpu"])
print("实验目录:", RUN)
if DEVICE.type != "cuda":
    print("未检测到 CUDA：正式训练会使用 CPU，请先检查内核和 GPU。")

## 1. 核对数据和基线

只读取现有元数据、YOLO 冻结记录和文件哈希；不会运行模型或访问测试图片。训练时会再次核对每个图片路径与类别。

In [ ]:
root, classes, rows, baseline_dir, yolo_test = experiment.data_info(PROJECT)
counts = {split: sum(row["split"] == split for row in rows) for split in ("train", "val", "test")}
print("数据:", root)
print("类别:", len(classes), "划分:", counts)
print("YOLO 基线实验:", baseline_dir)
print("YOLO 测试准确率:", f"{yolo_test['accuracy']:.4%}",
      "Macro-F1:", f"{yolo_test['macro_f1']:.4%}")

## 2. 检查网络

仅在 CPU 上执行一次全零输入的前向检查，不训练、不读取图片。论文未明确的通道宽度、瓶颈比例和 CBAM 参数采用实现中的固定选择。

In [ ]:
model = WMPeleeNet(num_classes=len(classes)).eval()
with torch.inference_mode():
    output = model(torch.zeros(1, 3, 224, 224))
assert output.shape == (1, 38) and torch.isfinite(output).all()
print("网络输出:", tuple(output.shape))
print("参数量:", f"{sum(p.numel() for p in model.parameters()):,}")
print("论文表中的模型约 169,000 参数；本重建实现并非同一份代码。")
del model, output

## 3. 开始训练

准备好后，把下面的 `RUN_TRAINING` 改成 `True`，运行本单元。实验目录必须是新的，不会覆盖已有实验。

每个完整轮次保存 `last.pt`，验证 Macro-F1 严格提高时保存 `best.pt`。中途中断后，保留同一 `RUN_TAG` 和配置，把 `RESUME_TRAINING` 改为 `True` 再运行；恢复从上一个完整轮次继续。如果还没完成第一轮，请更换 `RUN_TAG`。

In [ ]:
RUN_TRAINING = False
RESUME_TRAINING = False

if RUN_TRAINING:
    try:
        history = experiment.train(PROJECT, RUN, CONFIG, DEVICE, resume=RESUME_TRAINING)
        print("训练结束；最佳权重和每轮状态已保存到", RUN)
    except KeyboardInterrupt:
        print("训练已中断；可从 last.pt 恢复上一个完整轮次。")
else:
    print("训练未启动。设置 RUN_TRAINING=True 后手动运行本单元。")

## 4. 查看训练曲线

有检查点时读取已保存的训练历史，不继续训练。

In [ ]:
if (RUN / "last.pt").exists():
    saved = torch.load(RUN / "last.pt", map_location="cpu", weights_only=True)
    history_table = pd.DataFrame(saved["history"])
    display(history_table.tail())
    ax = history_table.plot(x="epoch", y=["val_accuracy", "val_macro_f1"], grid=True, figsize=(8, 3.5),
                            title="Validation accuracy and Macro-F1")
    ax.set_ylabel("Score")
else:
    print("尚无训练历史。")

## 5. 冻结最佳验证模型

正式训练完成或早停后，把 `FREEZE_MODEL` 改为 `True` 并运行。本步骤固定验证集最佳权重，冻结后不能在同一目录继续训练；需要重新实验时更换 `RUN_TAG`。

In [ ]:
FREEZE_MODEL = False
if FREEZE_MODEL:
    experiment.freeze(PROJECT, RUN, CONFIG)
    print("已冻结:", RUN / "frozen.pt")
else:
    print("模型未冻结。")

## 6. 测试集评估

确认模型已冻结后，把 `RUN_TEST` 改为 `True`。测试结果写入新实验目录；再次运行不会覆盖已有结果。

In [ ]:
RUN_TEST = False
if RUN_TEST:
    result = experiment.test(PROJECT, RUN, DEVICE, CONFIG["batch"])
    print("测试集样本:", result["samples"])
    print("准确率:", f"{result['accuracy']:.4%}", "Macro-F1:", f"{result['macro_f1']:.4%}")
else:
    print("测试未运行。")

## 7. 对比已有 YOLO26n 结果

两行使用同一份划分和类别映射；这是各一次实验的对照，初始化和部分训练细节不同。该单元只读取现有报告，不重新测试 YOLO。

In [ ]:
if (RUN / "test_metrics.json").exists() and (RUN / "frozen.json").exists():
    comparison = pd.DataFrame(experiment.compare(PROJECT, RUN))
    display(comparison)
else:
    print("WM-PeleeNet 测试完成后，这里会显示对照表。")

## 8. 测量两种模型的速度

冻结 WM 模型后，把 `RUN_BENCHMARK` 改为 `True`。两模型在**当前同一设备**、batch=1、相同预处理下测量；包含预处理、设备传输、模型推理、输出检查和同步，不含磁盘读取。会分别预热 20 次、计时 200 次。

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    speed = experiment.benchmark(PROJECT, RUN, DEVICE)
    display(pd.DataFrame({name: speed[name] for name in ("wm_pelee", "yolo26n")}).T)
    print("结果保存到", RUN / "speed.json")
else:
    print("速度测试未运行。")

## 输出与说明

`wm_peleenet_cls/runs/RUN_TAG/` 保存配置、每轮检查点、冻结权重、测试指标与速度报告。原始数据和 YOLO 文件只读。建议先看 Macro-F1、各类别报告，再结合本机推理速度判断。